# Practice Exercises: RAG & Vector Databases

**Post-class practice!** In this notebook, you will build your own mini RAG system step by step.

**Instructions:**
- Complete the lines marked with `# TODO` in each exercise
- Run the code and observe the output carefully
- Answer every "Reflection Question" in the markdown cell provided

**Setup:** Run this on Google Colab (Runtime → Change runtime type → GPU recommended, but CPU works too — just slower)


---
## Part 0: Setup

First, install and import the required libraries.


In [ ]:
# Run this first!
!pip install transformers torch sentence-transformers faiss-cpu -q

In [ ]:
from transformers import pipeline

# Load the text generation model (same as class)
generator = pipeline('text-generation', model='gpt2')

print("Setup complete! ✅")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setup complete! ✅


---
## Part 1: Hallucination vs RAG — The "Open Book Exam" 📖

Remember: without context, the LLM **guesses** (Hallucination). With RAG, we give it the "book" to read from.


### Exercise 1.1: Catch the Hallucination! 🕵️

**Task:** Ask GPT-2 a question about something it **cannot possibly know** (a made-up or very recent fact). Observe how it confidently makes something up.

Example ideas: *"Who won the 2027 Cricket World Cup?"*, *"What is the name of the mayor of Atlantis?"*


In [ ]:
# TODO: Write a question the model cannot know the answer to
my_question = "Who will won the men's odi world cup final 2027? "   # <-- your impossible question here

res = generator(my_question, max_new_tokens=30)
print(res[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Who will won the men's odi world cup final 2027?  We're going to see.  The final will be played at the Emirates Stadium on Saturday, October 1st 2017.  Here's


**🤔 Reflection Question 1.1:** Did the model say "I don't know"? Or did it confidently invent an answer? Why do you think LLMs hallucinate instead of admitting they don't know?

**Answer:**

The model did not say "I don't know." Instead, it generated an answer that sounded confident even though it had no reliable information. LLMs hallucinate because they are trained to predict the most likely next words based on patterns in their training data rather than checking whether a fact is true or whether they actually know the answer.


### Exercise 1.2: Fix It with RAG (Manual Augmentation) 🔧

**Task:** Now fix the hallucination from Exercise 1.1 using the **Augmentation** step:
1. Write a `private_knowledge` string that contains the correct answer to your question
2. Build a `rag_prompt` that combines **Context + Question** (like we did in class)
3. Compare the output with Exercise 1.1


In [ ]:
# TODO: Write the "ground truth" for your question
private_knowledge = "The 2027 men's odi world cup was won by India and their team score is 350 for 3 wickets down and Virat Kohli was the player of the match."   # <-- e.g., "The 2027 Cricket World Cup was won by ..."

# TODO: Build the RAG prompt — attach context to the question
# Format: Context: ... \n\n Question: ... \n Answer according to the context:
rag_prompt = f""" Context:{private_knowledge}
          Question: {my_question}
          Answer according to the context:
""" # <-- build your prompt here using private_knowledge and my_question

res_rag = generator(rag_prompt, max_new_tokens=50)
print(res_rag[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Context:The 2027 men's odi world cup was won by India and their team score is 350 for 3 wickets down and Virat Kohli was the player of the match. 
          Question: Who will won the men's odi world cup final 2027?  
          Answer according to the context:   

(1) India                               

(2) England        


**🤔 Reflection Question 1.2:** Which of the 3 RAG steps (Retrieval, Augmentation, Generation) did we do **manually** here? Which step is still missing from our system?

**Answer:**

In this exercise, we manually performed the Augmentation step by providing the correct information as context. The Generation step was done by the language model when it answered the question using that context. The Retrieval step is still missing because we manually wrote the knowledge instead of retrieving it automatically from a database or document.

---
## Part 2: Embeddings — Turning Meaning into Numbers 🔢

Remember: AI understands **Numbers**, not words. Similar meanings → vectors that are **close** on the graph.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model (same as class)
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready! ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready! ✅


### Exercise 2.1: Look Inside an Embedding 👀

**Task:** Convert a sentence into an embedding and inspect it:
1. Encode the sentence `"I love programming"`
2. Print the **shape** (how many numbers?) and the **first 10 numbers**


In [ ]:
sentence = "I love programming"

# TODO: encode the sentence using model.encode()
embedding = model.encode(sentence)   # <-- fix this

print("Shape of embedding:", embedding.shape)
print("First 10 numbers:", embedding[:10])

Shape of embedding: (384,)
First 10 numbers: [-0.03617867 -0.01277372  0.0030063  -0.01690344  0.00948426 -0.06515173
  0.09376633  0.07142349  0.01852631  0.05358271]


**🤔 Reflection Question 2.1:** How many dimensions (numbers) does one embedding have? Can a human read meaning from these numbers directly?

**Answer:**

One embedding has **384 dimensions (numbers)** when using the `all-MiniLM-L6-v2` Sentence Transformer model (your number may differ if you're using a different embedding model). Humans cannot directly understand the meaning of these numbers because they are mathematical representations of semantic information. Instead, machine learning models compare these vectors to measure the similarity and meaning between texts.


### Exercise 2.2: The King & Queen Test 👑

In class we said: *"King" and "Queen" vectors will be close, but "Apple" will be far away.* Let's **prove it with code**!

**Task:** Compute the similarity between sentence pairs. Higher score = closer meaning.


In [ ]:
from sentence_transformers import util

sentences = [
    "The king rules the country.",      # 0
    "The queen lives in the palace.",   # 1
    "I ate an apple for breakfast."     # 2
]

embeddings = model.encode(sentences)

# TODO: compute cosine similarity between sentence 0 and sentence 1 (king vs queen)
sim_king_queen = util.cos_sim(embeddings[0], embeddings[1])   # <-- fix the index

# TODO: compute cosine similarity between sentence 0 and sentence 2 (king vs apple)
sim_king_apple = util.cos_sim(embeddings[0], embeddings[2])   # <-- fix the index

print(f"King vs Queen similarity: {sim_king_queen.item():.4f}")
print(f"King vs Apple similarity: {sim_king_apple.item():.4f}")

King vs Queen similarity: 0.4106
King vs Apple similarity: 0.0614


**🤔 Reflection Question 2.2:** Which pair got the higher similarity score? Does this match the "close on the graph" logic from class?

**Answer:**

The **King vs Queen** pair got the higher similarity score because the two sentences are semantically related. Yes, this matches the "close on the graph" logic from class: embeddings with similar meanings are located closer together in the vector space, resulting in a higher cosine similarity score, while unrelated sentences like **King vs Apple** are farther apart and have a lower similarity score.



---
## Part 3: Build Your Own Vector Database 🗄️

Now the real deal — build a FAISS vector database with **your own documents** and search it.


### Exercise 3.1: Create Your Knowledge Base

**Task:** Write **5 documents of your own** (facts about your college, your city, your favorite topics — anything!). Then:
1. Encode them into vectors
2. Store them in a FAISS index
3. Print how many documents got stored


In [ ]:
import faiss

# TODO: Write your own 5 documents (make them about different topics!)
documents = [
    "Delhi is the capital of India.",
    "Python is widely used for Artificial Intelligence.",
    "Football is one of the world's most popular sports.",
    "The Pacific Ocean is the largest ocean.",
    "Photosynthesis helps plants make food."
]

# TODO: Convert documents into vectors using model.encode()
doc_embeddings = model.encode(documents)   # <-- fix this

# TODO: Build the FAISS index
dimension = doc_embeddings.shape[1]        # size of each vector
index = index = faiss.IndexFlatL2(dimension) # <-- create faiss.IndexFlatL2(...) here
index.add(np.array(doc_embeddings,dtype=np.float32))        # add vectors to the database

print(f"{index.ntotal} documents have been stored in the Vector Database.")

5 documents have been stored in the Vector Database.


### Exercise 3.2: Semantic Search Test 🔍

**Task:** Ask a question about one of YOUR documents — but **do NOT use the exact same words** as the document! This tests **Semantic Search** (meaning) vs **Keyword Search** (exact words).

Example: if your document says *"Almonds are healthy for the brain"*, ask *"Which food is good for memory?"*


In [ ]:
# TODO: Write a query that matches one document by MEANING, not exact words
query = "Tell me famous thing about Delhi"   # <-- your query

# TODO: Convert the query into a vector
query_embedding = model.encode([query])   # <-- fix (hint: model.encode([query]))

# TODO: Search the database for the top 1 result
D, I = index.search(np.array(query_embedding), k=1)   # <-- fix k

print(f"Question: {query}")
print(f"Most similar Document: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}  (lower = more similar)")

Question: Tell me famous thing about Delhi
Most similar Document: Delhi is the capital of India.
Distance: 0.5888  (lower = more similar)


**🤔 Reflection Question 3.2:** Did the database find the right document even though you used different words? Explain how this is different from Ctrl+F / keyword search:

**Answer:**

Yes. The vector database found the correct document even though the query used different words. Unlike Ctrl+F or keyword search, semantic search understands the meaning of the query by comparing embeddings rather than looking for exact word matches.


### Exercise 3.3: Top-K Retrieval

**Task:** Instead of only the top 1 result, retrieve the **top 3** most similar documents and print them in order with their distances.


In [ ]:
query2 = "Where do python used mostly?"   # <-- write another query

query_embedding2 = model.encode([query2])

# TODO: search with k=3
D, I = index.search(np.array(query_embedding2), k=3)   # <-- fix k

print(f"Question: {query2}\n")
# TODO: loop over the 3 results and print rank, document, and distance
for rank in range(3):   # <-- fix range
    print(f"Rank {rank+1}: {documents[I[0][rank]]}  (distance: {D[0][rank]:.4f})")

Question: Where do python used mostly?

Rank 1: Python is widely used for Artificial Intelligence.  (distance: 0.5147)
Rank 2: Football is one of the world's most popular sports.  (distance: 1.8312)
Rank 3: The Pacific Ocean is the largest ocean.  (distance: 1.9128)


**🤔 Reflection Question 3.3:** Look at the distances of ranks 1, 2, and 3. Is the rank-1 document clearly the best match, or are the distances close? When might retrieving more than 1 document be useful for RAG?

**Answer:**

The rank-1 document usually has the smallest distance, making it the best match. Retrieving more than one document is useful in RAG because multiple documents may contain complementary information that helps the LLM generate a more accurate answer.


---
## Part 4: 🏆 Final Challenge — Full RAG Pipeline (End-to-End)

Now connect **everything**: Retrieval (Vector DB) → Augmentation (prompt building) → Generation (LLM).

This is the complete flow from class, but built by YOU:

```
User Query → [Vector DB Search] → Best Document → [Attach to Prompt] → [GPT-2] → Answer
```


In [ ]:
def my_rag_pipeline(user_query):
    # STEP 1 — RETRIEVAL
    # TODO: encode the user_query into a vector
    q_emb = model.encode([user_query])   # <-- fix

    # TODO: search the FAISS index for the top 1 document
    D, I = index.search(np.array(q_emb), k=1)   # <-- fix
    retrieved_doc = documents[I[0][0]]

    # STEP 2 — AUGMENTATION
    # TODO: build the RAG prompt with the retrieved document as context
    rag_prompt = f"Context {retrieved_doc} Question {user_query} Answer according to the context"   # <-- Context: ... Question: ... Answer according to the context:

    # STEP 3 — GENERATION
    res = generator(rag_prompt, max_new_tokens=40)

    print("Retrieved Document:", retrieved_doc)
    print("-" * 60)
    print("Final Answer:\n", res[0]['generated_text'])

# Test your pipeline!
my_rag_pipeline("Which is the famous thing about Delhi?")   # <-- ask a question about one of your documents

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Document: Delhi is the capital of India.
------------------------------------------------------------
Final Answer:
 Context Delhi is the capital of India. Question Which is the famous thing about Delhi? Answer according to the context.

The Delhi government has a history of being very open to discussion and criticism. But that is where there is a problem. It is not because of the lack of a dedicated and competent police


**🤔 Final Reflection Question:**
1. In your pipeline, what would happen if the Vector DB retrieved the **wrong** document? Would the LLM still give a good answer?
2. Based on this, complete the sentence: *"A RAG system is only as good as its ______."*

Ans 1) If the Vector Database retrieves the wrong document, the LLM will use incorrect context and may generate an inaccurate or misleading answer. Therefore, good retrieval is essential for good RAG performance.

Ans 2) A **RAG** system is only as good as it's retriever.


### Bonus Challenge 4.1: RAG vs Fine-tuning Decision Table 🎓

For each scenario below, decide: **RAG or Fine-tuning?** (Remember: Fine-tuning = 5 years of med school, RAG = textbook in the exam)

| Scenario                                                                       | RAG / Fine-tuning? | Reason                                                                                              |
| ------------------------------------------------------------------------------ | ------------------ | --------------------------------------------------------------------------------------------------- |
| A company chatbot that must answer from HR policy PDFs that change every month | **RAG**            | Policies change frequently, so retrieving the latest documents is better than retraining the model. |
| Teaching a model to always respond in Shakespearean English style              | **Fine-tuning**    | This changes the model's writing style, which requires learning new behavior.                       |
| A news assistant that must know today's headlines                              | **RAG**            | News changes every day, so retrieving fresh information is necessary.                               |
| A medical model that must deeply understand doctor-style reasoning             | **Fine-tuning**    | This requires teaching the model specialized reasoning and domain knowledge.                        |


### Bonus Challenge 4.2: Break the Retriever! 💥

**Task:** Try to find a query where your Vector DB retrieves the **WRONG** document.

Hints to try:
- A very vague query (e.g., "Tell me something")
- A query about a topic NOT in any of your 5 documents
- A query mixing two topics at once

Print the result and the distance score.


In [ ]:
# Try to fool the retriever
tricky_query = "Tell me something "

q_emb = model.encode([tricky_query])

D, I = index.search(np.array(q_emb).astype("float32"), k=1)

print(f"Question: {tricky_query}")
print(f"Retrieved: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}")

Question: Tell me something 
Retrieved: Football is one of the world's most popular sports.
Distance: 1.7975


**🤔 Reflection Question 4.2:** What happened when you asked about a topic that was NOT in your database? Did the retriever say "not found", or did it still return *something*? Why is this a problem for real RAG systems, and how might we solve it? (Hint: think about the distance score!)

**Answer:**

The retriever still returned the closest document even though none of the documents were about the FIFA World Cup. It does not automatically return "not found." This is a problem because the LLM may generate an answer using irrelevant context. A common solution is to set a similarity (or distance) threshold so that if no document is similar enough, the system responds with "I don't know" or asks the user for more information instead.

---
## ✅ Submission Checklist

Before submitting, check:

- [ ] Part 1: Hallucination caught + fixed with manual RAG
- [ ] Part 2: Embedding inspected + King/Queen similarity test done
- [ ] Part 3: Your own 5-document Vector DB built + semantic search + top-3 retrieval
- [ ] Part 4: Full end-to-end RAG pipeline working
- [ ] Bonus 4.1 table filled in
- [ ] All "Reflection Questions" answered
- [ ] **All cells have been run** (outputs are visible)

**Well done! 🎉 You have built a complete RAG system from scratch!**
